# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² mental health survey dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj) from Kilifi County, Kenya using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined via the Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

We'll reference all dataset entities by their `@id` as prescribed by the Croissant specification.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# `metadata` is an mlcroissant.metadata.DatasetMetadata object
print(f"Loaded dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}  |  Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Next, let's review available record sets, their fields, and the associated `@id`s.

**Note:** The mlcroissant API standardizes with `record_set` and `field` objects, each with their own `@id`.

* Print the list of available record sets (`@id`, name, description)
* For the first record set, print all its fields (`@id`, name, data type, description)
* We'll use these `@id`s in all subsequent calls.

In [ ]:
# List available record sets with their @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")
for idx, rs in enumerate(record_sets):
    print(f"RecordSet {idx+1}: @id={rs.id}\n  Name: {rs.name}\n  Description: {rs.description or 'No description.'}\n")

# Let's show the fields for the primary data RecordSet (usually the main survey table)
# We'll use the first RecordSet for demo (replace index if main data is at another position)
main_recordset = record_sets[0]
print(f"Fields for RecordSet '@id={main_recordset.id}':\n")
for f in main_recordset.fields:
    print(f"  Field @id={f.id}\n    Name: {f.name}\n    Data type: {f.data_type}\n    Description: {f.description or 'No description.'}\n")

## 3. Data Extraction
We'll now load one or more record sets into Pandas DataFrames for analysis.

- We reference each record set and field by their `@id`
- All further data filtering and analytics will use these DataFrames

**Note:** For large datasets, consider streaming with `.records()` rather than loading all records at once.

In [ ]:
# --- Parameterize record set @id(s) ---
record_set_ids = [rs.id for rs in record_sets]
# e.g. ['#KilifiSurveyMainTable'] or full IRIs. Use what you printed above.
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet '@id={record_set_id}' ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records with {len(df.columns)} fields: {df.columns.tolist()[:10]}{'...' if len(df.columns) > 10 else ''}")
    else:
        print("  (No records found)")
print("\nAvailable DataFrames (by RecordSet @id):", list(dataframes.keys()))

# Let's display the first few rows of the main record set
main_df_key = record_set_ids[0]  # Use the first RecordSet's @id
main_df = dataframes[main_df_key]
print(f"First rows of main DataFrame (RecordSet @id={main_df_key}):")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data processing and inspection steps:
- Filter: For example, select respondents with PHQ-9 total score above a threshold (replace the field `@id` accordingly)
- Normalize: Z-score normalization for a numeric field
- Group: Calculate means by some categorical attribute (e.g., gender/age group)

All field choices are by `@id` (see above listing).

In [ ]:
# --- SET these IDs to match the survey field "@id" values as printed. ---
# You may need to adjust these to the real field @id(s) for PHQ-9 total and gender.

NUMERIC_FIELD_ID = None
GROUP_FIELD_ID = None
# Find field ids for 'phq9_total' and 'gender' (search by name if unsure)
for f in main_recordset.fields:
    name = f.name.lower() if f.name else ""
    if 'phq' in name and 'total' in name:
        NUMERIC_FIELD_ID = f.id
    elif 'gend' in name:
        GROUP_FIELD_ID = f.id
print(f"Using fields: NUMERIC_FIELD_ID={NUMERIC_FIELD_ID}  GROUP_FIELD_ID={GROUP_FIELD_ID}")

# If field ids are not found, print DataFrame columns for reference
if NUMERIC_FIELD_ID not in main_df.columns or GROUP_FIELD_ID not in main_df.columns:
    print(f"Available columns in main_df: {list(main_df.columns)}")

# Filtering, normalization, and grouping
if NUMERIC_FIELD_ID in main_df.columns:
    # Only keep numeric rows
    _df = main_df.copy()
    # Convert to numeric as Croissant record fields are often all string
    _df[NUMERIC_FIELD_ID] = pd.to_numeric(_df[NUMERIC_FIELD_ID], errors='coerce')
    threshold = 10
    filtered_df = _df[_df[NUMERIC_FIELD_ID] > threshold]
    print(f"Records with {NUMERIC_FIELD_ID} > {threshold}: {len(filtered_df)}")
    # Normalization
    filtered_df[f"{NUMERIC_FIELD_ID}_zscore"] = (filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()) / filtered_df[NUMERIC_FIELD_ID].std()
    print(filtered_df[[NUMERIC_FIELD_ID, f"{NUMERIC_FIELD_ID}_zscore"]].head())
    # Group statistics
    if GROUP_FIELD_ID and GROUP_FIELD_ID in filtered_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().reset_index()
        print("\nMean PHQ-9 by group:")
        print(grouped_df.head())
else:
    print(f"Numeric field '{NUMERIC_FIELD_ID}' not found in DataFrame columns.")

## 5. Visualization
Let's visualize the distribution for the numeric field (e.g., PHQ-9 total scores) and display a simple group bar plot. 

Adjust the visualization as needed based on the actual available fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if NUMERIC_FIELD_ID and NUMERIC_FIELD_ID in main_df.columns:
    # Histogram of the PHQ-9 total field
    plt.figure(figsize=(7, 4))
    main_df[NUMERIC_FIELD_ID] = pd.to_numeric(main_df[NUMERIC_FIELD_ID], errors='coerce')
    sns.histplot(main_df[NUMERIC_FIELD_ID].dropna(), bins=12, kde=True, color='skyblue')
    plt.title(f"Distribution of {NUMERIC_FIELD_ID}")
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if GROUP_FIELD_ID and GROUP_FIELD_ID in main_df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=main_df[GROUP_FIELD_ID], y=main_df[NUMERIC_FIELD_ID])
        plt.title(f"{NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}")
        plt.xlabel(GROUP_FIELD_ID)
        plt.ylabel(NUMERIC_FIELD_ID)
        plt.show()
else:
    print("Numeric field for visualization not available.")

## 6. Conclusion
We have explored the FAIR² dataset with `mlcroissant`, loaded and visualized its main records, normalized and grouped a numeric mental health score, and produced basic visual summaries.

To go further, you can join with more record sets (if present), add quality checks for missing values, or build predictive models using the Croissant schema for consistent, reproducible workflows.

**References:**  [FAIR² Dataset on SenScience](https://sen.science/doi/10.71728/senscience.vcs2-05nj)